# RMIT_SGS_Sigma69_C1 — V3, học từ notebook Melbourne

**Backbone mặc định: `patronus-studio/wolf-defender-prompt-injection` (ModernBERT),
fine-tune toàn bộ**, từ Kaggle `aisuko/wolf-defender-prompt-injection/transformers/default/1`.

Chạy trên **Colab GPU trong runtime mới**. Upload ZIP dữ liệu ở mục 3 hoặc dùng Drive.
Notebook độc lập; không cần upload các script trong repository.

**Kết quả đã có:** DistilBERT V2 raw validation Macro F1 `0.919711`,
cross-fit postprocess `0.952069`, fitted validation `0.958402`.
Notebook Melbourne dùng Wolf Defender/ModernBERT: raw validation `0.960082`,
fitted bias `0.964711`; public `0.97` do người dùng cung cấp.
Code Melbourne ghi 4 epoch nhưng output lưu 2 epoch, nên đây chưa phải bản tái lập chính xác.

**V3:** chọn một backbone bằng `MODEL_PRESET`; mở rộng global bias theo Melbourne,
chỉ dùng hiệu chỉnh nếu qua cross-fit; giữ checkpoint theo raw Macro F1;
sau đó tùy chọn train lại từ pretrained trên đủ 14.000 mẫu với số epoch đã chọn.
`USE_FULL_TRAIN=True` tốn thêm một lượt train. Full refit không có validation độc lập;
mọi báo cáo validation thuộc mô hình ở bước chọn cấu hình.

`distilbert`: QLoRA r64, 8 epoch tối đa, focal loss, head-tail 512 token.
`wolf`: full fine-tune ModernBERT, weighted CE, 4 epoch tối đa, 256 token như Melbourne.
Wolf đã được đối chiếu trong danh sách được phép trên trang Challenge của BTC, 22/10/2026.
Hai backbone không được train/blend cùng nhau. TF-IDF + SVM chỉ học từ train.csv.

Mục tiêu **0.99 chưa được kiểm chứng**; chưa chạy GPU bản V3.
Bản V2 đã chạy được lưu nguyên vẹn tại `notebooks/archive/` trong repo.
Cuối notebook tạo `/content/submission.csv` và ZIP artifacts. Tải CSV bằng panel Files;
ZIP chứa model/adapter và báo cáo, không phải prediction file để nộp Kaggle.


## 1. Cài dependencies

Giữ bộ Transformers/PEFT/bitsandbytes của lần chạy cũ và pin scikit-learn theo baseline.
Nếu cell yêu cầu restart, chỉ restart sau khi đã tải/lưu artifacts của phiên cũ; chạy lại từ đầu.


In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

PINS = {
    "transformers": "4.57.6",
    "peft": "0.17.1",
    "accelerate": "1.10.1",
    "bitsandbytes": "0.48.1",
    "scikit-learn": "1.6.1",
}
loaded_conflicts = []
for package, wanted in PINS.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    module_name = "sklearn" if package == "scikit-learn" else package
    if module_name in sys.modules and installed != wanted:
        loaded_conflicts.append(package)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    *[f"{name}=={release}" for name, release in PINS.items()],
    "huggingface-hub>=0.34,<1",
    "pandas>=2.2,<3", "joblib>=1.2,<2",
    "matplotlib>=3.8,<4", "safetensors>=0.4.3", "kagglehub>=0.3.13,<1",
])
if loaded_conflicts:
    raise RuntimeError(
        "Da doi thu vien dang duoc import: " + ", ".join(loaded_conflicts)
        + ". Chon Runtime > Restart session, sau do chay lai tu dau."
    )
print("Dependencies ready. PyTorch CUDA cua Colab duoc giu nguyen.")


## 2. Cấu hình

Giữ **DATA_MODE = "upload"** để chọn CSV/ZIP từ máy. Nếu dữ liệu đã ở Colab, chọn **"path"** và sửa DATA_DIR. Muốn dùng Google Drive, bật MOUNT_DRIVE và đặt DATA_DIR hoặc OUTPUT_ROOT dưới thư mục Drive.

TRAIN_BATCH_SIZE × GRAD_ACCUM_STEPS = effective batch size. Nếu hết VRAM, giảm batch từ 8 xuống 4 và tăng accumulation từ 4 lên 8. LONG_TEXT_MODE có thể đổi từ **"head"** sang **"head_tail"** cho một thí nghiệm riêng.


In [ ]:
# Chon MOT backbone. Chay lai tu dau khi doi preset, khong tai dung logits cu.
MODEL_PRESET = "wolf"                       # "wolf" hoac "distilbert"; chi MOT backbone
WOLF_APPROVAL_CONFIRMED = True              # Challenge page 877551, 22/10/2026
WOLF_APPROVAL_SOURCE = "https://www.kaggle.com/competitions/rmit-hackathon-2026/overview/challenge"
WOLF_KAGGLE_HANDLE = "aisuko/wolf-defender-prompt-injection/transformers/default/1"
WOLF_LOCAL_PATH = ""                        # Tuy chon: folder model da tai/gan tren Kaggle

DATA_MODE = "upload"                       # "upload" hoac "path"
DATA_DIR = "/content/rmit-hackathon-2026"
MOUNT_DRIVE = False
OUTPUT_ROOT = "/content/rmit_c1_runs"        # Doi sang Drive de giu checkpoints
SEED = 42
SPLIT_SEED = 42
VAL_SIZE = 0.10
USE_FULL_TRAIN = True                      # 90/10 selection -> fresh refit tren 14k
FULL_TRAIN_EPOCHS = None                   # None = epoch cua checkpoint tot nhat
TRAIN_BATCH_SIZE = 8                       # OOM: giam 4, tang GRAD_ACCUM_STEPS len 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
USE_CLASS_WEIGHTS = True
GRADIENT_CHECKPOINTING = True

if MODEL_PRESET == "distilbert":
    MODEL_SOURCE = "distilbert/distilbert-base-multilingual-cased"
    MODEL_REVISION = "45c032ab32cc946ad88a166f7cb282f58c753c2e"
    EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE = 8, 5e-5, 3
    MAX_LENGTH, LONG_TEXT_MODE = 512, "head_tail"
    CLASS_WEIGHT_POWER, FOCAL_GAMMA = 0.65, 2.0
elif MODEL_PRESET == "wolf":
    if not WOLF_APPROVAL_CONFIRMED or not WOLF_APPROVAL_SOURCE.strip():
        raise ValueError("Can doi chieu whitelist BTC cho Wolf va ghi WOLF_APPROVAL_SOURCE.")
    MODEL_SOURCE = WOLF_LOCAL_PATH          # Tu tai dung model version 1 neu de trong
    MODEL_REVISION = None
    EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE = 4, 3e-5, 2
    MAX_LENGTH, LONG_TEXT_MODE = 256, "head"
    CLASS_WEIGHT_POWER, FOCAL_GAMMA = 0.5, 0.0  # Weighted CE de bat dau gan Melbourne
else:
    raise ValueError("MODEL_PRESET phai la distilbert hoac wolf.")
RUN_NAME = f"{MODEL_PRESET}_v3"

# Cac tham so LoRA chi dung khi MODEL_PRESET == "distilbert".
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
LORA_TARGETS = ["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"]
HEAD_MODULES = ["pre_classifier", "classifier"]
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}
LANGUAGES = ["pt", "fr", "it", "en"]
ENABLE_LEXICAL_BRANCH = True
LEXICAL_MAX_FEATURES, LEXICAL_C, LEXICAL_GLOBAL_WEIGHT = 300000, 2.0, 0.5
MIN_POSTPROCESS_GAIN = 0.002
# Giu cac moc bias nho V2 va them range rong Melbourne; KHONG copy bias da fit cua ho.
POSTPROCESS_FIT_KWARGS = {
    "jailbreak_bias_values": [-1., -.75, -.5, -.3, -.15, 0., .15, .3, .5, .75, 1.],
    "injection_bias_values": [-1., -.75, -.5, -.3, -.15, 0., .15, .3, .5, .75, 1., 1.5, 2.],
    "bias_l2": 0.002,
}
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATA_MODE in {"upload", "path"}
assert LONG_TEXT_MODE in {"head", "head_tail"}
assert 0 < VAL_SIZE < 1
assert 8 <= MAX_LENGTH <= 512 and MAX_LENGTH % 8 == 0
assert TRAIN_BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0
assert EPOCHS >= 1 and 0 <= CLASS_WEIGHT_POWER <= 1 and FOCAL_GAMMA >= 0
assert 0 <= LEXICAL_GLOBAL_WEIGHT <= 1
print("Preset:", MODEL_PRESET, "| full refit:", USE_FULL_TRAIN)
print("Effective batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)


In [ ]:
import gc
import io
import json
import hashlib
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    AutoConfig, AutoTokenizer, PreTrainedTokenizerFast, AutoModelForSequenceClassification,
    BitsAndBytesConfig, DataCollatorWithPadding, Trainer,
    TrainingArguments, EarlyStoppingCallback, set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from IPython.display import display, FileLink

if not torch.cuda.is_available():
    raise RuntimeError("Can GPU CUDA: Runtime > Change runtime type > GPU, roi chay lai.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Notebook yeu cau mot GPU. Restart runtime de CUDA_VISIBLE_DEVICES co hieu luc.")

set_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# T4 (capability 7.5) khong co native BF16, du PyTorch co the cho phep emulation.
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
PACKAGE_VERSIONS = {
    name: version(name)
    for name in ["torch", "transformers", "peft", "accelerate", "bitsandbytes",
                 "huggingface-hub", "numpy", "pandas", "scikit-learn", "safetensors", "joblib", "kagglehub"]
}
for package, wanted in PINS.items():
    if PACKAGE_VERSIONS[package] != wanted:
        raise RuntimeError(f"Sai version {package}; restart runtime va chay lai cell cai dat.")

DATA_PATH = Path(DATA_DIR)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = Path(OUTPUT_ROOT) / f"{RUN_NAME}_{RUN_ID}"
ARTIFACT_DIR = RUN_DIR / "artifacts"
ADAPTER_DIR = ARTIFACT_DIR / ("selection_model" if MODEL_PRESET == "wolf" else "selection_adapter")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)

print("Python:", sys.version)
print("GPU capability:", GPU_CAPABILITY)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Compute dtype:", COMPUTE_DTYPE)
print("Output:", RUN_DIR)
print("Training preset:", MODEL_PRESET)
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["package", "version"]))


## 3. Upload dữ liệu

Chọn cả ba CSV trong một lần hoặc upload ZIP từ Kaggle. Nếu dùng ZIP, notebook chỉ lấy ba CSV có tên đúng, kể cả khi chúng nằm trong thư mục con. Không cần Kaggle token và không tải dữ liệu từ GitHub.


In [ ]:
EXPECTED_FILES = {"train.csv", "test.csv", "sample_submission.csv"}

if DATA_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    selected = {}

    def add_csv(name, payload):
        if name in selected:
            raise ValueError(f"Trung file {name} trong lan upload. Chi chon mot ban moi file.")
        selected[name] = payload

    for upload_name, payload in uploaded.items():
        filename = Path(upload_name).name
        if filename in EXPECTED_FILES:
            add_csv(filename, payload)
        elif filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                for member in archive.infolist():
                    name = Path(member.filename).name
                    if not member.is_dir() and name in EXPECTED_FILES:
                        add_csv(name, archive.read(member))

    missing = EXPECTED_FILES - set(selected)
    if missing:
        raise ValueError(f"Upload thieu {sorted(missing)}. Chay lai cell va chon ca ba CSV/ZIP.")
    DATA_PATH.mkdir(parents=True, exist_ok=True)
    for name, payload in selected.items():
        (DATA_PATH / name).write_bytes(payload)
    del uploaded, selected

missing = [name for name in EXPECTED_FILES if not (DATA_PATH / name).is_file()]
if missing:
    raise FileNotFoundError(f"Khong tim thay {missing} trong {DATA_PATH}. Kiem tra DATA_DIR.")
print("Data directory:", DATA_PATH)


In [ ]:
def load_csv(name):
    # Giu nguyen text va ID, khong bien chuoi "NA" thanh gia tri thieu.
    return pd.read_csv(DATA_PATH / name, dtype=str, keep_default_na=False, encoding="utf-8-sig")

train_df = load_csv("train.csv")
test_df = load_csv("test.csv")
sample_df = load_csv("sample_submission.csv")

def validate_frame(frame, name, columns, expected_rows):
    if list(frame.columns) != columns:
        raise ValueError(f"{name}: can cot {columns}; nhan duoc {list(frame.columns)}.")
    if len(frame) != expected_rows:
        raise ValueError(f"{name}: can {expected_rows} dong; nhan duoc {len(frame)}.")
    if frame["id"].isna().any() or frame["id"].str.strip().eq("").any() or not frame["id"].is_unique:
        raise ValueError(f"{name}: ID rong hoac trung.")
    for column in ["text", "language"]:
        if column in frame and (
            frame[column].isna().any() or frame[column].str.strip().eq("").any()
        ):
            raise ValueError(f"{name}: {column} rong.")
    if "language" in frame and not set(frame["language"]).issubset(LANGUAGES):
        raise ValueError(f"{name}: language ngoai mien {LANGUAGES}.")

validate_frame(train_df, "train.csv", ["id", "text", "language", "label"], 14000)
validate_frame(test_df, "test.csv", ["id", "text", "language"], 6000)
validate_frame(sample_df, "sample_submission.csv", ["id", "label"], 6000)
if set(train_df["label"]) != set(LABELS):
    raise ValueError("Train phai chua du ba nhan chinh xac.")
if set(sample_df["id"]) != set(test_df["id"]):
    raise ValueError("ID sample_submission khong khop test.")
if train_df["text"].duplicated().any():
    raise ValueError("Train co text trung; can group-aware split de tranh leakage.")

train_df["label_id"] = train_df["label"].map(LABEL2ID).astype("int64")
DATA_HASHES = {
    name: hashlib.sha256((DATA_PATH / name).read_bytes()).hexdigest()
    for name in sorted(EXPECTED_FILES)
}
display(pd.crosstab(train_df["language"], train_df["label"]).reindex(columns=LABELS))
display(train_df["text"].str.len().describe().rename("text_characters").to_frame())
print("Train:", len(train_df), "| Test:", len(test_df))
print("Sample labels chi la placeholder; khong dung lam ground truth.")


## 4. Holdout để chọn checkpoint và class weights

Luôn chia 90/10 theo language × label, kể cả khi `USE_FULL_TRAIN=True`.
Chỉ bước refit cuối mới dùng toàn bộ train; không dùng validation đã học để chọn model.
Class weights được tính riêng trên tập train của từng bước.


In [ ]:
strata = train_df["language"] + "::" + train_df["label"]
smallest_group = int(strata.value_counts().min())
if smallest_group < 2:
    raise ValueError("Nhom language/label qua nho; can xem lai cach split.")

fit_df, valid_df = train_test_split(
    train_df, test_size=VAL_SIZE, random_state=SPLIT_SEED, stratify=strata,
)
fit_df = fit_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
assert set(fit_df["id"]).isdisjoint(set(valid_df["id"]))
assert set(fit_df["text"]).isdisjoint(set(valid_df["text"]))
assert set(fit_df["id"]) | set(valid_df["id"]) == set(train_df["id"])

split_manifest = pd.concat([
    fit_df[["id", "language", "label"]].assign(split="train"),
    valid_df[["id", "language", "label"]].assign(split="validation"),
], ignore_index=True)
split_manifest.to_csv(ARTIFACT_DIR / "split.csv", index=False)

def compute_class_weights(label_ids, num_classes, power=0.65, enabled=True):
    ids = np.asarray(label_ids, dtype=np.int64)
    if ids.ndim != 1 or ids.size == 0 or not np.isin(ids, np.arange(num_classes)).all():
        raise ValueError("Label IDs khong hop le.")
    if not 0 <= power <= 1:
        raise ValueError("Class weight power phai trong [0, 1].")
    counts = np.bincount(ids, minlength=num_classes)
    if (counts == 0).any():
        raise ValueError("Moi class phai co mau train.")
    values = (len(ids) / counts.astype(np.float64)) ** (power if enabled else 0.0)
    values /= np.average(values, weights=counts)
    return values.astype(np.float32)

class_counts = np.bincount(fit_df["label_id"].to_numpy(), minlength=len(LABELS))
power = CLASS_WEIGHT_POWER if USE_CLASS_WEIGHTS else 0.0
weight_values = compute_class_weights(fit_df["label_id"], len(LABELS), power, USE_CLASS_WEIGHTS)
class_weights = torch.tensor(weight_values, dtype=torch.float32)
print("Selection training:", len(fit_df), "| Validation:", len(valid_df))
display(pd.DataFrame({"label": LABELS, "train_count": class_counts, "weight": weight_values}))


### Nhánh bổ trợ TF-IDF ký tự + SVM

Học prompt không cắt ngắn, gồm classifier chung và classifier theo ngôn ngữ.
Ở selection, chỉ fit trên 12.600 mẫu; validation/test chỉ transform/predict.
Ở full refit, fit lại trên 14.000 mẫu có nhãn. Không fit trên test.csv.
Global/local cố định 50/50; cross-fit chọn mức phối hợp với backbone.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
import joblib

def fit_lexical_branch(frame):
    # Chi fit tren frame co nhan cua tung stage; khong fit tren test.
    vectorizer = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 6), min_df=2,
        max_features=LEXICAL_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
    )
    features = vectorizer.fit_transform(frame["text"].tolist())
    y = frame["label_id"].to_numpy()
    languages = frame["language"].to_numpy()

    def fit_classifier(x, labels):
        classifier = LinearSVC(
            C=LEXICAL_C, class_weight="balanced", dual="auto",
            max_iter=5000, random_state=SEED,
        )
        classifier.fit(x, labels)
        assert np.array_equal(classifier.classes_, np.arange(len(LABELS)))
        return classifier

    global_model = fit_classifier(features, y)
    language_models = {}
    for language in LANGUAGES:
        mask = languages == language
        if set(y[mask]) == set(range(len(LABELS))):
            language_models[language] = fit_classifier(features[mask], y[mask])
    return {"vectorizer": vectorizer, "global_model": global_model,
            "language_models": language_models, "labels": list(LABELS),
            "global_weight": LEXICAL_GLOBAL_WEIGHT}

def lexical_predict(bundle, frame):
    assert bundle["labels"] == LABELS, "Sai thu tu nhan lexical model."
    features = bundle["vectorizer"].transform(frame["text"].tolist())
    global_scores = bundle["global_model"].decision_function(features)
    local_scores = global_scores.copy()
    for language, classifier in bundle["language_models"].items():
        mask = frame["language"].eq(language).to_numpy()
        if mask.any():
            local_scores[mask] = classifier.decision_function(features[mask])
    weight = bundle["global_weight"]
    scores = weight * global_scores + (1.0 - weight) * local_scores
    assert scores.shape == (len(frame), len(LABELS)) and np.isfinite(scores).all()
    return scores.astype(np.float32)

valid_lexical_scores = test_lexical_scores = None
lexical_bundle = None
if ENABLE_LEXICAL_BRANCH:
    lexical_bundle = fit_lexical_branch(fit_df)
    valid_lexical_scores = lexical_predict(lexical_bundle, valid_df)
    test_lexical_scores = lexical_predict(lexical_bundle, test_df)
    lexical_f1 = f1_score(
        valid_df["label_id"], valid_lexical_scores.argmax(axis=1),
        labels=np.arange(len(LABELS)), average="macro", zero_division=0,
    )
    print("Lexical validation Macro F1:", round(lexical_f1, 6))
    print("Features:", len(lexical_bundle["vectorizer"].vocabulary_))
    # Can bundle nay + adapter + postprocess.json de tai lap ensemble inference.
    joblib.dump(lexical_bundle, ARTIFACT_DIR / "selection_lexical.joblib", compress=3)
    np.save(ARTIFACT_DIR / "validation_lexical_scores.npy", valid_lexical_scores)
    np.save(ARTIFACT_DIR / "test_lexical_scores.npy", test_lexical_scores)
else:
    print("Lexical branch disabled; submission se dung backbone da chon.")


## 5. Tokenizer và prompt dài

Wolf mặc định lấy đầu prompt trong 256 token theo Melbourne; DistilBERT giữ đầu–cuối trong 512 token.
Tokenizer tự thêm token đặc biệt trước khi cắt; giữ nguyên các token này ở cả hai chế độ.
Có thể thử head-tail/512 cho Wolf trong lần chạy riêng và so sánh holdout.
Lưu tỷ lệ bị cắt theo language/label; không thêm prefix ngôn ngữ vào đầu vào backbone.


In [ ]:
# Wolf dung dung Kaggle model/version trong notebook Melbourne, khong thay bang repo gan ten.
if MODEL_PRESET == "wolf":
    if not MODEL_SOURCE:
        import kagglehub
        try:
            MODEL_SOURCE = kagglehub.model_download(WOLF_KAGGLE_HANDLE)
        except Exception as exc:
            raise RuntimeError(
                "Khong tai duoc Wolf tu KaggleHub. Kiem tra Internet/Kaggle login, "
                "hoac dat WOLF_LOCAL_PATH toi folder model co config.json."
            ) from exc
    model_root = Path(MODEL_SOURCE)
    candidates = ([model_root / "config.json"] if (model_root / "config.json").is_file()
                  else list(model_root.rglob("config.json")))
    if len(candidates) != 1:
        raise ValueError("WOLF_LOCAL_PATH phai tro dung mot model folder co config.json.")
    MODEL_SOURCE = str(candidates[0].parent)

base_config = AutoConfig.from_pretrained(
    MODEL_SOURCE, revision=MODEL_REVISION, trust_remote_code=False,
)
expected_type = "modernbert" if MODEL_PRESET == "wolf" else "distilbert"
if base_config.model_type != expected_type:
    raise ValueError(f"Preset {MODEL_PRESET} can model_type={expected_type}.")
if MAX_LENGTH > base_config.max_position_embeddings:
    raise ValueError("MAX_LENGTH vuot gioi han position embeddings.")

def prepare_wolf_v457_compatibility(config):
    """Map the V5 Wolf RoPE schema onto pinned Transformers 4.57.6 fields."""
    rope = getattr(config, "rope_parameters", None)
    try:
        full_theta = float(rope["full_attention"]["rope_theta"])
        local_theta = float(rope["sliding_attention"]["rope_theta"])
    except (KeyError, TypeError, ValueError) as exc:
        raise ValueError("Wolf config thieu rope_parameters cua checkpoint da kiem tra.") from exc
    config.global_rope_theta = full_theta
    config.local_rope_theta = local_theta
    layer_types = getattr(config, "layer_types", None)
    if layer_types is not None:
        interval = int(config.global_attn_every_n_layers)
        expected = [
            "full_attention" if index % interval == 0 else "sliding_attention"
            for index in range(int(config.num_hidden_layers))
        ]
        if list(layer_types) != expected:
            raise ValueError("Wolf layer_types khong tuong thich voi ModernBERT 4.57.6.")
    return config

if MODEL_PRESET == "wolf":
    base_config = prepare_wolf_v457_compatibility(base_config)

RESOLVED_MODEL_REVISION = getattr(base_config, "_commit_hash", None)
LOAD_KWARGS = {"trust_remote_code": False}
if MODEL_PRESET == "distilbert":
    LOAD_KWARGS["revision"] = RESOLVED_MODEL_REVISION or MODEL_REVISION
MODEL_PROVENANCE = {
    "preset": MODEL_PRESET, "source": MODEL_SOURCE,
    "revision": LOAD_KWARGS.get("revision"),
    "kaggle_handle": WOLF_KAGGLE_HANDLE if MODEL_PRESET == "wolf" else None,
    "approval_source": WOLF_APPROVAL_SOURCE if MODEL_PRESET == "wolf" else "docs/requirements.md",
}
if MODEL_PRESET == "wolf":
    # Wolf v1 was exported by Transformers 5 and declares TokenizersBackend,
    # a class unavailable in the pinned 4.57.6. Load tokenizer.json directly.
    tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_SOURCE, **LOAD_KWARGS)
    if not tokenizer.is_fast:
        raise ValueError("Wolf tokenizer phai la fast tokenizer.")
    for name in ("pad", "bos", "eos", "cls", "sep"):
        tokenizer_id = getattr(tokenizer, f"{name}_token_id")
        config_id = getattr(base_config, f"{name}_token_id")
        if tokenizer_id != config_id:
            raise ValueError(f"Wolf tokenizer/config lech {name}_token_id: {tokenizer_id} != {config_id}")
    probe = tokenizer("prompt test", add_special_tokens=True)["input_ids"]
    if base_config.cls_token_id not in probe or base_config.sep_token_id not in probe:
        raise ValueError("Wolf tokenizer khong chen dung CLS/SEP.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True, **LOAD_KWARGS)

def encode_texts(texts):
    # Let the fast tokenizer's real postprocessor insert CLS/SEP. The generic
    # PreTrainedTokenizerFast.build_inputs_with_special_tokens does not do so.
    encoded = tokenizer(
        texts, add_special_tokens=True, truncation=False,
        return_special_tokens_mask=True, return_attention_mask=False,
        return_token_type_ids=False,
    )
    final_ids, lengths = [], []
    for token_ids, special_mask in zip(encoded["input_ids"], encoded["special_tokens_mask"]):
        positions = [i for i, special in enumerate(special_mask) if not special]
        lengths.append(len(positions))
        budget = MAX_LENGTH - sum(special_mask)
        if budget < 1:
            raise ValueError("MAX_LENGTH khong du cho special tokens va noi dung.")
        if len(positions) > budget:
            if LONG_TEXT_MODE == "head":
                chosen = positions[:budget]
            else:
                head_count = (budget + 1) // 2
                tail_count = budget - head_count
                chosen = positions[:head_count] + (positions[-tail_count:] if tail_count else [])
            kept = set(chosen)
            token_ids = [token_id for i, token_id in enumerate(token_ids) if special_mask[i] or i in kept]
        assert len(token_ids) <= MAX_LENGTH
        final_ids.append(token_ids)
    return {"input_ids": final_ids, "attention_mask": [[1] * len(ids) for ids in final_ids]}, lengths

class PromptDataset(Dataset):
    def __init__(self, frame, labelled):
        self.features = {"input_ids": [], "attention_mask": []}
        self.labels = frame["label_id"].tolist() if labelled else None
        self.raw_lengths = []
        for start in range(0, len(frame), 256):
            encoded, lengths = encode_texts(frame["text"].iloc[start:start + 256].tolist())
            self.raw_lengths.extend(lengths)
            for key in self.features:
                self.features[key].extend(encoded[key])

    def __len__(self):
        return len(self.features["input_ids"])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.features.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[index])
        return item

train_dataset = PromptDataset(fit_df, labelled=True)
valid_dataset = PromptDataset(valid_df, labelled=True)
test_dataset = PromptDataset(test_df, labelled=False)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
budget = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
length_reports = []
for split, frame, dataset in [("train", fit_df, train_dataset),
                              ("validation", valid_df, valid_dataset),
                              ("test", test_df, test_dataset)]:
    report = frame[["id", "language"]].copy()
    report["label"] = frame["label"].to_numpy() if "label" in frame else "UNKNOWN"
    report["split"] = split
    report["raw_tokens"] = dataset.raw_lengths
    report["truncated"] = report["raw_tokens"] > budget
    length_reports.append(report)
token_lengths = pd.concat(length_reports, ignore_index=True)
token_lengths.to_csv(ARTIFACT_DIR / "token_lengths.csv", index=False)
display(token_lengths.groupby(["split", "language", "label"])["truncated"].agg(["size", "sum", "mean"]))
print("Datasets:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("Model revision:", RESOLVED_MODEL_REVISION or MODEL_REVISION)


## 6. Khởi tạo backbone đã chọn

Wolf dùng full fine-tune, head 3 lớp được khởi tạo lại, SDPA và AMP; tắt compile nội bộ
ModernBERT để chạy đơn giản trên T4. DistilBERT dùng QLoRA NF4 và train cả hai lớp head.
Factory luôn khởi tạo mới từ pretrained cho cả selection và full refit.


In [ ]:
def build_model():
    """Reload the approved source and reset the seed for selection or final refit."""
    from copy import deepcopy

    set_seed(SEED)
    if MODEL_PRESET == "wolf":
        if base_config.model_type != "modernbert":
            raise ValueError("Preset wolf yeu cau checkpoint Wolf-Defender ModernBERT.")
        config = deepcopy(base_config)
        config.num_labels = len(LABELS)
        config.id2label = dict(ID2LABEL)
        config.label2id = dict(LABEL2ID)
        config.problem_type = "single_label_classification"
        # ModernBERT co torch.compile noi bo, doc lap voi Trainer.torch_compile.
        # Dat lai False moi lan load: ModernBertConfig.to_dict() bo truong nay.
        config.reference_compile = False
        loaded = AutoModelForSequenceClassification.from_pretrained(
            MODEL_SOURCE,
            config=config,
            # Checkpoint Wolf co 2 nhan; khoi tao lai classifier cho 3 nhan C1.
            ignore_mismatched_sizes=True,
            dtype=torch.float32,
            attn_implementation="sdpa",
            use_safetensors=True,
            **LOAD_KWARGS,
        )
        assert loaded.config.model_type == "modernbert"
        assert loaded.config.reference_compile is False
        assert loaded.classifier.out_features == len(LABELS)
        assert not getattr(loaded, "is_loaded_in_4bit", False)
        # Full fine-tuning: FP32 parameters; Trainer bat AMP FP16/BF16.
        assert all(parameter.requires_grad for parameter in loaded.parameters())
        assert all(parameter.dtype == torch.float32 for parameter in loaded.parameters())
        print("Wolf-Defender: full fine-tuning, 3-label classifier, SDPA, reference_compile=False.")
    elif MODEL_PRESET == "distilbert":
        if base_config.model_type != "distilbert":
            raise ValueError("Preset distilbert yeu cau DistilBERT multilingual.")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            # Transformers 4.57.6 su dung skip_modules cho ca 4-bit quantizer.
            llm_int8_skip_modules=list(HEAD_MODULES),
        )
        loaded = AutoModelForSequenceClassification.from_pretrained(
            MODEL_SOURCE,
            num_labels=len(LABELS),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            problem_type="single_label_classification",
            quantization_config=quant_config,
            device_map={"": 0},
            dtype=COMPUTE_DTYPE,
            attn_implementation="sdpa",
            use_safetensors=True,
            **LOAD_KWARGS,
        )
        assert getattr(loaded, "is_loaded_in_4bit", False), "Backbone chua load 4-bit."
        assert any(isinstance(module, bnb.nn.Linear4bit) for module in loaded.modules())
        for name in HEAD_MODULES:
            assert not isinstance(getattr(loaded, name), bnb.nn.Linear4bit), f"{name} khong duoc quantize."
        loaded = prepare_model_for_kbit_training(
            loaded,
            use_gradient_checkpointing=GRADIENT_CHECKPOINTING,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
        for suffix in LORA_TARGETS:
            hits = [name for name, module in loaded.named_modules() if name.endswith("." + suffix)]
            assert len(hits) == base_config.n_layers, (suffix, hits)
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,
            modules_to_save=list(HEAD_MODULES),
            bias="none",
        )
        loaded = get_peft_model(loaded, lora_config)
        trainable_names = [name for name, parameter in loaded.named_parameters() if parameter.requires_grad]
        assert trainable_names
        assert any("lora_" in name for name in trainable_names)
        for head in HEAD_MODULES:
            assert any(f".{head}.modules_to_save." in name for name in trainable_names)
        assert all(
            "lora_" in name or any(f".{head}.modules_to_save." in name for head in HEAD_MODULES)
            for name in trainable_names
        ), "Co tham so backbone dang train ngoai adapter/head."
        loaded.print_trainable_parameters()
    else:
        raise ValueError(f"MODEL_PRESET khong hop le: {MODEL_PRESET}")

    assert loaded.config.num_labels == len(LABELS)
    assert loaded.config.id2label == ID2LABEL
    trainable_names = [name for name, parameter in loaded.named_parameters() if parameter.requires_grad]
    trainable_count = sum(parameter.numel() for parameter in loaded.parameters() if parameter.requires_grad)
    total_count = sum(parameter.numel() for parameter in loaded.parameters())
    print(f"Trainable: {trainable_count:,} / {total_count:,} ({100 * trainable_count / total_count:.2f}%)")
    (ARTIFACT_DIR / "trainable_parameters.txt").write_text(
        "\n".join(trainable_names) + "\n", encoding="utf-8",
    )
    return loaded


model = build_model()


## 7. Macro F1 và loss

Weighted focal loss tính `p_t` từ cross-entropy **chưa nhân class weight**, giữ gradient
qua hệ số focal. `gamma=0` trở về weighted CE. Mẫu số là tổng số mẫu trong bước
gradient accumulation, kể cả microbatch cuối nhỏ hơn. Wolf mặc định CE như Melbourne;
DistilBERT mặc định focal gamma 2 theo kế hoạch. Không suy ra gain trước khi chạy.


In [ ]:
def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    expected = prediction.label_ids
    class_f1 = f1_score(
        expected, predicted, labels=list(range(len(LABELS))),
        average=None, zero_division=0,
    )
    result = {
        "macro_f1": float(class_f1.mean()),
        "accuracy": float(accuracy_score(expected, predicted)),
    }
    result.update({f"f1_{label.lower()}": float(score) for label, score in zip(LABELS, class_f1)})
    return result

def make_weighted_focal_loss(weights, gamma):
    if gamma < 0:
        raise ValueError("Focal gamma phai >= 0.")
    # Bind weights rieng cho tung Trainer; full refit khong doi closure cua selection.
    fixed_weights = torch.as_tensor(weights, dtype=torch.float32).clone()

    def weighted_focal_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits.float()
        ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        alpha = fixed_weights.to(device=logits.device, dtype=logits.dtype)[labels]
        losses = alpha * (1.0 - pt).pow(gamma) * ce
        count = labels.numel() if num_items_in_batch is None else num_items_in_batch
        denominator = torch.as_tensor(count, device=logits.device, dtype=logits.dtype).clamp_min(1)
        return losses.sum() / denominator

    return weighted_focal_loss


def selected_checkpoint_epoch(state):
    # early stopping co the ket thuc muon hon checkpoint tot nhat.
    import math
    checkpoint = getattr(state, "best_model_checkpoint", None)
    if not checkpoint:
        raise ValueError("Thieu best_model_checkpoint de chon epoch refit.")
    try:
        step = int(str(checkpoint).replace("\\", "/").rstrip("/").rsplit("checkpoint-", 1)[1])
    except (ValueError, IndexError) as exc:
        raise ValueError("Khong doc duoc step cua best checkpoint.") from exc
    matches = [row for row in state.log_history
               if row.get("step") == step and "eval_macro_f1" in row and "epoch" in row]
    if not matches:
        raise ValueError("Khong co eval log khop best checkpoint.")
    epoch = float(matches[-1]["epoch"])
    if not math.isfinite(epoch) or epoch <= 0:
        raise ValueError("Epoch cua checkpoint khong hop le.")
    return int(math.ceil(epoch - 1e-9))


weighted_focal_loss = make_weighted_focal_loss(class_weights, FOCAL_GAMMA)


In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    run_name=RUN_NAME,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    label_names=["labels"],
    remove_unused_columns=True,
    eval_accumulation_steps=16,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    push_to_hub=False,
    torch_compile=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_loss_func=weighted_focal_loss,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
# DistilBERT forward khong nhan num_items_in_batch; loss callback xu ly tham so nay.
trainer.model_accepts_loss_kwargs = False
print("Trainer ready. Cell tiep theo bat dau huan luyen tren Colab.")


## 8. Huấn luyện và chọn checkpoint

Wolf train tối đa 4 epoch; DistilBERT tối đa 8 epoch. Early stopping theo raw Macro F1,
sau đó lấy checkpoint có Macro F1 cao nhất. Training loss giảm không đảm bảo điểm test tăng.
Hết VRAM: giảm batch 8 → 4 và tăng accumulation 4 → 8, giữ effective batch 32.
Lưu OUTPUT_ROOT trên Drive để giữ checkpoint khi Colab ngắt phiên.


In [ ]:
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation Macro F1:", trainer.state.best_metric)

# Luu selection model (Wolf) hoac adapter + heads (DistilBERT) truoc refit.
trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.model.config.to_json_file(str(ARTIFACT_DIR / "base_config.json"))
print("Selection weights saved:", ADAPTER_DIR)


## 9. Đánh giá model tốt nhất

Báo cáo Macro F1, F1 từng lớp, theo ngôn ngữ và confusion matrix đều dùng validation chưa tham gia cập nhật trọng số. Kết quả này là holdout validation; chưa phải điểm Kaggle.


In [ ]:
valid_result = trainer.predict(valid_dataset, metric_key_prefix="validation")
valid_logits = valid_result.predictions
if isinstance(valid_logits, tuple):
    valid_logits = valid_logits[0]
if not np.isfinite(valid_logits).all():
    raise ValueError("Validation logits co NaN/Inf; kiem tra training.")
valid_pred = np.argmax(valid_logits, axis=-1)
valid_true = valid_df["label_id"].to_numpy()
report = classification_report(
    valid_true, valid_pred, labels=list(range(len(LABELS))),
    target_names=LABELS, output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report).T
display(report_df)
report_df.to_csv(ARTIFACT_DIR / "validation_class_report.csv")

language_rows = []
for language in LANGUAGES:
    mask = valid_df["language"].eq(language).to_numpy()
    if not mask.any():
        continue
    f1_values = f1_score(
        valid_true[mask], valid_pred[mask], labels=list(range(len(LABELS))),
        average=None, zero_division=0,
    )
    row = {"language": language, "rows": int(mask.sum()), "macro_f1": float(f1_values.mean())}
    row.update({f"f1_{label}": float(score) for label, score in zip(LABELS, f1_values)})
    language_rows.append(row)
language_report = pd.DataFrame(language_rows)
display(language_report)
language_report.to_csv(ARTIFACT_DIR / "validation_by_language.csv", index=False)

validation_predictions = valid_df[["id", "language", "label"]].copy()
validation_predictions["predicted_label"] = [ID2LABEL[int(value)] for value in valid_pred]
validation_predictions.to_csv(ARTIFACT_DIR / "validation_predictions.csv", index=False)

matrix = confusion_matrix(valid_true, valid_pred, labels=list(range(len(LABELS))))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(
    ax=ax, values_format="d", cmap="Blues", xticks_rotation=30, colorbar=False,
)
ax.set_title("Validation confusion matrix")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "validation_confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)

np.save(ARTIFACT_DIR / "validation_logits.npy", valid_logits.astype(np.float32))
valid_df[["id", "language", "label"]].to_csv(ARTIFACT_DIR / "validation_ids.csv", index=False)
raw_errors = valid_df[["id", "language", "label", "text"]].copy()
raw_errors["predicted_label"] = [ID2LABEL[int(i)] for i in valid_pred]
raw_errors["raw_tokens"] = valid_dataset.raw_lengths
raw_errors["truncated"] = raw_errors["raw_tokens"] > budget
raw_errors.loc[valid_true != valid_pred].to_csv(ARTIFACT_DIR / "validation_errors_raw.csv", index=False)


## 10. Dự đoán test và xuất submission

Dự đoán giữ liên kết với test ID, sau đó sắp theo **sample_submission.csv**. File nộp chỉ có hai cột **id,label** và đúng 6.000 dòng. Nhãn mẫu không ảnh hưởng dự đoán.


In [ ]:
test_result = trainer.predict(test_dataset, metric_key_prefix="test")
test_logits = test_result.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
if test_logits.shape != (len(test_df), len(LABELS)) or not np.isfinite(test_logits).all():
    raise ValueError("Test logits sai shape hoac co NaN/Inf.")
np.save(ARTIFACT_DIR / "test_logits.npy", test_logits.astype(np.float32))
test_df[["id", "language"]].to_csv(ARTIFACT_DIR / "test_ids.csv", index=False)
print("Da luu logits va ID order. Cell sau chon ensemble/bias tu validation.")


## 10. Ensemble và global bias theo hướng Melbourne

Tìm bias JAILBREAK trong [-1, 1], PROMPT_INJECTION trong [-1, 2], giữ thêm các mức nhỏ V2.
Không chép bias `[-1, -0.75]` đã fit của Melbourne sang mô hình này.
Cross-fit 3 fold trên validation; chỉ bật nếu gain ít nhất `MIN_POSTPROCESS_GAIN`.
Không thêm bias theo ngôn ngữ vì nhóm English/PROMPT_INJECTION chỉ khoảng 10 mẫu validation.
Macro F1 fitted trên toàn bộ validation là số dùng chọn cấu hình, không phải điểm test.


In [ ]:
"""Conservative post-processing cell for the RMIT C1 notebook.

Paste this file into one notebook code cell after ``valid_logits`` and
``test_logits`` exist. If the lexical SVM is enabled, its decision-function
outputs must be named ``valid_lexical_scores`` and ``test_lexical_scores`` and
must already use the same class order as ``LABELS``.

This cell deliberately performs no artifact I/O. The notebook's artifact cell
should save the returned configuration, report, predictions, and source logits.
"""

from __future__ import annotations

from itertools import product
from typing import Any, Iterable, Mapping, Optional, Sequence

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold


EXPECTED_POSTPROCESS_LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]


def _as_score_matrix(name: str, values: Any, rows: Optional[int] = None) -> np.ndarray:
    """Return a finite float64 (rows, 3) score matrix."""
    matrix = np.asarray(values, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != len(EXPECTED_POSTPROCESS_LABELS):
        raise ValueError(
            f"{name} must have shape (n, {len(EXPECTED_POSTPROCESS_LABELS)}); "
            f"received {matrix.shape}."
        )
    if rows is not None and matrix.shape[0] != rows:
        raise ValueError(f"{name} must have {rows} rows; received {matrix.shape[0]}.")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} contains NaN or Inf.")
    return matrix


def log_softmax_np(scores: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Numerically stable log-softmax for a small dense score matrix."""
    temperature = float(temperature)
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature must be finite and positive.")
    scaled = np.asarray(scores, dtype=np.float64) / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))


def softmax_from_log_scores(log_scores: np.ndarray) -> np.ndarray:
    """Normalize arbitrary log-scores into probabilities."""
    values = np.asarray(log_scores, dtype=np.float64)
    shifted = values - values.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return probabilities


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray, class_count: int = 3) -> float:
    """Exact unweighted macro F1, including classes with zero support."""
    truth = np.asarray(y_true, dtype=np.int64)
    predicted = np.asarray(y_pred, dtype=np.int64)
    if truth.shape != predicted.shape or truth.ndim != 1:
        raise ValueError("Expected aligned one-dimensional labels.")
    if not np.isin(truth, np.arange(class_count)).all() or not np.isin(predicted, np.arange(class_count)).all():
        raise ValueError("Class index outside allowed range.")
    matrix = np.bincount(class_count * truth + predicted, minlength=class_count ** 2).reshape(class_count, class_count)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    values = np.divide(2.0 * matrix.diagonal(), denominator,
                       out=np.zeros(class_count, dtype=np.float64), where=denominator > 0)
    return float(values.mean())


def postprocess_log_scores(
    neural_logits: np.ndarray,
    lexical_scores: Optional[np.ndarray],
    config: Mapping[str, Any],
) -> np.ndarray:
    """Apply a fitted temperature/blend/bias configuration."""
    neural_log_proba = log_softmax_np(
        neural_logits, float(config["neural_temperature"])
    )
    lexical_weight = float(config.get("lexical_weight", 0.0))
    if lexical_weight > 0:
        if lexical_scores is None:
            raise ValueError("The fitted configuration requires lexical scores.")
        lexical_log_proba = log_softmax_np(
            lexical_scores, float(config["lexical_temperature"])
        )
        combined = (
            (1.0 - lexical_weight) * neural_log_proba
            + lexical_weight * lexical_log_proba
        )
    else:
        combined = neural_log_proba

    bias = np.asarray(config.get("class_bias", [0.0, 0.0, 0.0]), dtype=np.float64)
    if bias.shape != (neural_log_proba.shape[1],) or not np.isfinite(bias).all():
        raise ValueError("class_bias has an invalid shape or value.")
    return combined + bias[None, :]


def _config_simplicity_key(config: Mapping[str, Any]) -> tuple[Any, ...]:
    """Prefer no bias, then neural-only, then values closest to identity."""
    bias = np.asarray(config["class_bias"], dtype=np.float64)
    bias_norm = float(np.dot(bias, bias))
    lexical_weight = float(config["lexical_weight"])
    lexical_temperature = config.get("lexical_temperature")
    return (
        int(bias_norm > 1e-15),
        int(lexical_weight > 1e-15),
        bias_norm,
        lexical_weight,
        abs(float(np.log(float(config["neural_temperature"])))),
        0.0
        if lexical_temperature is None
        else abs(float(np.log(float(lexical_temperature)))),
    )


def fit_postprocessor(
    neural_logits: np.ndarray,
    y_true: np.ndarray,
    lexical_scores: Optional[np.ndarray] = None,
    *,
    neural_temperatures: Sequence[float] = (1.0, 2.0),
    lexical_temperatures: Sequence[float] = (0.5, 1.0),
    lexical_weights: Sequence[float] = (0.0, 0.25, 0.50, 0.75, 0.90),
    bias_values: Sequence[float] = (-0.30, -0.15, 0.0, 0.15, 0.30),
    jailbreak_bias_values: Optional[Sequence[float]] = None,
    injection_bias_values: Optional[Sequence[float]] = None,
    bias_l2: float = 0.002,
    tie_tolerance: float = 1e-12,
) -> dict[str, Any]:
    """Fit a small global post-processing grid on the supplied rows.

    BENIGN bias is fixed to zero because adding the same constant to all class
    scores cannot change a prediction. Only two bias coordinates are searched.
    Candidate matrices are evaluated one at a time to keep memory bounded.
    """
    neural_logits = _as_score_matrix("neural_logits", neural_logits)
    y_true = np.asarray(y_true, dtype=np.int64)
    if y_true.shape != (neural_logits.shape[0],):
        raise ValueError("y_true length does not match neural_logits.")
    if not np.isin(y_true, np.arange(neural_logits.shape[1])).all():
        raise ValueError("y_true contains a class outside [0, 2].")
    if lexical_scores is not None:
        lexical_scores = _as_score_matrix(
            "lexical_scores", lexical_scores, neural_logits.shape[0]
        )

    neural_temperatures = tuple(float(value) for value in neural_temperatures)
    lexical_temperatures = tuple(float(value) for value in lexical_temperatures)
    lexical_weights = tuple(float(value) for value in lexical_weights)
    bias_values = tuple(float(value) for value in bias_values)
    jailbreak_grid = tuple(bias_values if jailbreak_bias_values is None else jailbreak_bias_values)
    injection_grid = tuple(bias_values if injection_bias_values is None else injection_bias_values)
    if any(0.0 not in grid or not np.isfinite(grid).all() for grid in (jailbreak_grid, injection_grid)):
        raise ValueError("Moi bias grid can 0 va cac gia tri huu han.")
    if 1.0 not in neural_temperatures or 0.0 not in lexical_weights or 0.0 not in bias_values:
        raise ValueError("The conservative grid must include temperature=1, weight=0, and bias=0.")
    if any(value <= 0 for value in neural_temperatures + lexical_temperatures):
        raise ValueError("All temperatures must be positive.")
    if any(value < 0 or value >= 1 for value in lexical_weights):
        raise ValueError("lexical_weights must be in [0, 1).")
    if bias_l2 < 0:
        raise ValueError("bias_l2 must be nonnegative.")

    neural_cache = {
        temperature: log_softmax_np(neural_logits, temperature)
        for temperature in neural_temperatures
    }
    lexical_cache = (
        {
            temperature: log_softmax_np(lexical_scores, temperature)
            for temperature in lexical_temperatures
        }
        if lexical_scores is not None
        else {}
    )

    best_config: Optional[dict[str, Any]] = None
    best_objective = -np.inf
    best_score = -np.inf
    candidate_count = 0

    for neural_temperature in neural_temperatures:
        neural_log_proba = neural_cache[neural_temperature]
        weights_to_try: Iterable[float] = (
            lexical_weights if lexical_scores is not None else (0.0,)
        )
        for lexical_weight in weights_to_try:
            temperatures_to_try: Iterable[Optional[float]] = (
                (None,) if lexical_weight == 0.0 else lexical_temperatures
            )
            for lexical_temperature in temperatures_to_try:
                if lexical_weight == 0.0:
                    base_scores = neural_log_proba
                else:
                    lexical_log_proba = lexical_cache[float(lexical_temperature)]
                    base_scores = (
                        (1.0 - lexical_weight) * neural_log_proba
                        + lexical_weight * lexical_log_proba
                    )

                for jailbreak_bias, injection_bias in product(jailbreak_grid, injection_grid):
                    bias = np.array([0.0, jailbreak_bias, injection_bias], dtype=np.float64)
                    predictions = np.argmax(base_scores + bias[None, :], axis=1)
                    score = macro_f1(y_true, predictions, base_scores.shape[1])
                    bias_norm = float(np.dot(bias, bias))
                    objective = score - float(bias_l2) * bias_norm
                    config = {
                        "neural_temperature": float(neural_temperature),
                        "lexical_temperature": (
                            None if lexical_temperature is None else float(lexical_temperature)
                        ),
                        "lexical_weight": float(lexical_weight),
                        "class_bias": bias.tolist(),
                    }
                    candidate_count += 1

                    is_better = objective > best_objective + tie_tolerance
                    is_tied = abs(objective - best_objective) <= tie_tolerance
                    if is_better or (
                        is_tied
                        and (
                            best_config is None
                            or _config_simplicity_key(config)
                            < _config_simplicity_key(best_config)
                        )
                    ):
                        best_config = config
                        best_objective = float(objective)
                        best_score = float(score)

    if best_config is None:
        raise RuntimeError("Post-processing grid unexpectedly produced no candidate.")
    best_config.update(
        {
            "fit_macro_f1": best_score,
            "penalized_objective": best_objective,
            "bias_l2": float(bias_l2),
            "candidate_count": int(candidate_count),
        }
    )
    return best_config


def cross_fitted_postprocess(
    neural_logits: np.ndarray,
    y_true: np.ndarray,
    strata: np.ndarray,
    lexical_scores: Optional[np.ndarray] = None,
    *,
    n_splits: int = 3,
    random_state: int = 42,
    fit_kwargs: Optional[Mapping[str, Any]] = None,
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Select post-processing on two folds and predict the held-out fold."""
    neural_logits = _as_score_matrix("neural_logits", neural_logits)
    y_true = np.asarray(y_true, dtype=np.int64)
    strata = np.asarray(strata, dtype=str)
    if y_true.shape != (neural_logits.shape[0],) or strata.shape != y_true.shape:
        raise ValueError("y_true/strata length does not match neural_logits.")
    if lexical_scores is not None:
        lexical_scores = _as_score_matrix(
            "lexical_scores", lexical_scores, neural_logits.shape[0]
        )

    unique_strata, stratum_counts = np.unique(strata, return_counts=True)
    if len(unique_strata) < neural_logits.shape[1]:
        raise ValueError("Too few strata for cross-fitted post-processing.")
    if int(stratum_counts.min()) < int(n_splits):
        raise ValueError(
            f"Every language::label stratum needs at least {n_splits} rows; "
            f"smallest has {int(stratum_counts.min())}."
        )

    splitter = StratifiedKFold(
        n_splits=int(n_splits), shuffle=True, random_state=int(random_state)
    )
    oof_predictions = np.full(y_true.shape, -1, dtype=np.int64)
    fold_reports: list[dict[str, Any]] = []
    fit_kwargs = dict(fit_kwargs or {})

    for fold, (fit_index, holdout_index) in enumerate(
        splitter.split(np.zeros(y_true.shape[0]), strata), start=1
    ):
        fit_lexical = None if lexical_scores is None else lexical_scores[fit_index]
        holdout_lexical = None if lexical_scores is None else lexical_scores[holdout_index]
        config = fit_postprocessor(
            neural_logits[fit_index],
            y_true[fit_index],
            fit_lexical,
            **fit_kwargs,
        )
        holdout_log_scores = postprocess_log_scores(
            neural_logits[holdout_index], holdout_lexical, config
        )
        holdout_predictions = np.argmax(holdout_log_scores, axis=1)
        oof_predictions[holdout_index] = holdout_predictions
        raw_predictions = np.argmax(neural_logits[holdout_index], axis=1)
        raw_score = macro_f1(y_true[holdout_index], raw_predictions)
        post_score = macro_f1(y_true[holdout_index], holdout_predictions)
        fold_reports.append(
            {
                "fold": int(fold),
                "fit_rows": int(fit_index.size),
                "holdout_rows": int(holdout_index.size),
                "holdout_raw_neural_macro_f1": raw_score,
                "holdout_postprocess_macro_f1": post_score,
                "holdout_gain": float(post_score - raw_score),
                "selected_config": config,
            }
        )

    if (oof_predictions < 0).any():
        raise RuntimeError("Some validation rows did not receive a cross-fitted prediction.")
    return oof_predictions, fold_reports


def run_postprocessing(
    valid_logits: np.ndarray,
    test_logits: np.ndarray,
    valid_df: Any,
    labels: Sequence[str],
    valid_lexical_scores: Optional[np.ndarray] = None,
    test_lexical_scores: Optional[np.ndarray] = None,
    *,
    split_seed: int = 42,
    min_postprocess_gain: float = 0.002,
    n_splits: int = 3,
    fit_kwargs: Optional[Mapping[str, Any]] = None,
) -> dict[str, Any]:
    """Run cross-fitted selection, its safety gate, and final all-valid fit."""
    if list(labels) != EXPECTED_POSTPROCESS_LABELS:
        raise ValueError(
            f"LABELS must be exactly {EXPECTED_POSTPROCESS_LABELS}; received {list(labels)}."
        )
    valid_logits = _as_score_matrix("valid_logits", valid_logits, len(valid_df))
    test_logits = _as_score_matrix("test_logits", test_logits)
    if min_postprocess_gain < 0:
        raise ValueError("min_postprocess_gain must be nonnegative.")

    lexical_enabled = valid_lexical_scores is not None or test_lexical_scores is not None
    if lexical_enabled and (valid_lexical_scores is None or test_lexical_scores is None):
        raise ValueError(
            "valid_lexical_scores and test_lexical_scores must both be provided or both be None."
        )
    if lexical_enabled:
        valid_lexical_scores = _as_score_matrix(
            "valid_lexical_scores", valid_lexical_scores, valid_logits.shape[0]
        )
        test_lexical_scores = _as_score_matrix(
            "test_lexical_scores", test_lexical_scores, test_logits.shape[0]
        )

    required_columns = {"id", "language", "label_id"}
    missing_columns = required_columns - set(valid_df.columns)
    if missing_columns:
        raise ValueError(f"valid_df is missing columns: {sorted(missing_columns)}.")
    if valid_df["id"].duplicated().any():
        raise ValueError("valid_df IDs must be unique.")
    y_true = valid_df["label_id"].to_numpy(dtype=np.int64)
    if not np.isin(y_true, np.arange(len(labels))).all():
        raise ValueError("valid_df.label_id contains a class outside [0, 2].")
    languages = valid_df["language"].astype(str).to_numpy()
    strata = np.asarray(
        [f"{language}::{label_id}" for language, label_id in zip(languages, y_true)],
        dtype=str,
    )

    raw_valid_pred = np.argmax(valid_logits, axis=1)
    raw_macro = macro_f1(y_true, raw_valid_pred, len(labels))
    oof_predictions, fold_reports = cross_fitted_postprocess(
        valid_logits,
        y_true,
        strata,
        valid_lexical_scores,
        n_splits=n_splits,
        random_state=split_seed,
        fit_kwargs=fit_kwargs,
    )
    oof_macro = macro_f1(y_true, oof_predictions, len(labels))
    oof_gain = float(oof_macro - raw_macro)
    gate_passed = bool(oof_gain >= float(min_postprocess_gain))

    shared_validation_warning = (
        "Cross-fitting protects the post-processing comparison from fitting and scoring each "
        "bias/blend on the same rows. It is still not an independent model estimate because "
        "the checkpoint and other hyperparameters were selected using this validation set."
    )
    all_valid_fit_warning = (
        "The all-validation tuned Macro F1 is an in-sample diagnostic only; use the "
        "cross-fitted score for the post-processing gate."
    )

    if gate_passed:
        fitted_config = fit_postprocessor(
            valid_logits,
            y_true,
            valid_lexical_scores,
            **dict(fit_kwargs or {}),
        )
        final_valid_log_scores = postprocess_log_scores(
            valid_logits, valid_lexical_scores, fitted_config
        )
        final_test_log_scores = postprocess_log_scores(
            test_logits, test_lexical_scores, fitted_config
        )
        final_valid_proba = softmax_from_log_scores(final_valid_log_scores)
        final_test_proba = softmax_from_log_scores(final_test_log_scores)
        final_valid_pred = np.argmax(final_valid_proba, axis=1)
        final_test_pred = np.argmax(final_test_proba, axis=1)
        full_valid_macro = macro_f1(y_true, final_valid_pred, len(labels))
        postprocess_config = {
            "enabled": True,
            "mode": "global_log_probability_blend_and_bias",
            "lexical_enabled": bool(lexical_enabled),
            "minimum_oof_gain": float(min_postprocess_gain),
            "parameters": fitted_config,
        }
    else:
        final_valid_log_scores = log_softmax_np(valid_logits, 1.0)
        final_test_log_scores = log_softmax_np(test_logits, 1.0)
        final_valid_proba = np.exp(final_valid_log_scores)
        final_test_proba = np.exp(final_test_log_scores)
        final_valid_pred = raw_valid_pred.copy()
        final_test_pred = np.argmax(test_logits, axis=1)
        full_valid_macro = raw_macro
        postprocess_config = {
            "enabled": False,
            "mode": "raw_neural",
            "lexical_enabled": bool(lexical_enabled),
            "minimum_oof_gain": float(min_postprocess_gain),
            "parameters": {
                "neural_temperature": 1.0,
                "lexical_temperature": None,
                "lexical_weight": 0.0,
                "class_bias": [0.0, 0.0, 0.0],
            },
        }

    selection_report = {
        "gate_passed": gate_passed,
        "lexical_enabled": bool(lexical_enabled),
        "validation_rows": int(valid_logits.shape[0]),
        "cross_fit_folds": int(n_splits),
        "raw_neural_macro_f1": raw_macro,
        "cross_fitted_postprocess_macro_f1": oof_macro,
        "cross_fitted_gain": oof_gain,
        "minimum_required_gain": float(min_postprocess_gain),
        "all_validation_tuned_macro_f1": full_valid_macro,
        "folds": fold_reports,
        "warnings": [shared_validation_warning, all_valid_fit_warning],
        "language_specific_bias": False,
    }

    return {
        "final_valid_pred": final_valid_pred,
        "final_test_pred": final_test_pred,
        "final_valid_proba": final_valid_proba,
        "final_test_proba": final_test_proba,
        "final_valid_scores": final_valid_log_scores,
        "final_test_scores": final_test_log_scores,
        "postprocess_oof_pred": oof_predictions,
        "POSTPROCESS_CONFIG": postprocess_config,
        "POSTPROCESS_SELECTION_REPORT": selection_report,
    }


# When pasted as a notebook cell, publish the requested variables. Importing
# this file as a module does not execute the notebook-specific branch.
_required_notebook_names = {"valid_logits", "test_logits", "valid_df", "LABELS"}
if _required_notebook_names.issubset(globals()):
    MIN_POSTPROCESS_GAIN = float(globals().get("MIN_POSTPROCESS_GAIN", 0.002))
    POSTPROCESS_SPLIT_SEED = int(
        globals().get("SPLIT_SEED", globals().get("SEED", 42))
    )
    _postprocess_result = run_postprocessing(
        valid_logits=globals()["valid_logits"],
        test_logits=globals()["test_logits"],
        valid_df=globals()["valid_df"],
        labels=globals()["LABELS"],
        valid_lexical_scores=globals().get("valid_lexical_scores"),
        test_lexical_scores=globals().get("test_lexical_scores"),
        split_seed=POSTPROCESS_SPLIT_SEED,
        min_postprocess_gain=MIN_POSTPROCESS_GAIN,
        n_splits=3,
        fit_kwargs=POSTPROCESS_FIT_KWARGS,
    )
    globals().update(_postprocess_result)
    print(
        "Post-processing:",
        "enabled" if POSTPROCESS_CONFIG["enabled"] else "raw neural retained",
        "| raw Macro F1 =",
        round(POSTPROCESS_SELECTION_REPORT["raw_neural_macro_f1"], 6),
        "| cross-fitted Macro F1 =",
        round(POSTPROCESS_SELECTION_REPORT["cross_fitted_postprocess_macro_f1"], 6),
        "| gain =",
        round(POSTPROCESS_SELECTION_REPORT["cross_fitted_gain"], 6),
    )

POSTPROCESS_SELECTION_REPORT["raw_lexical_macro_f1"] = (
    macro_f1(valid_true, valid_lexical_scores.argmax(axis=1))
    if valid_lexical_scores is not None else None
)
POSTPROCESS_CONFIG["labels"] = list(LABELS)
POSTPROCESS_CONFIG["score_interpretation"] = (
    "Normalized tempered log-score blend, tuned for Macro F1; not calibrated confidence."
)
POSTPROCESS_CONFIG["search_grid"] = {
    "neural_temperatures": [1.0, 2.0], "lexical_temperatures": [0.5, 1.0],
    "lexical_weights": [0.0, 0.25, 0.50, 0.75, 0.90],
    **POSTPROCESS_FIT_KWARGS,
}
print("Final parameters:", POSTPROCESS_CONFIG["parameters"])


### Xuất CSV và báo cáo của bước selection

`submission_selection.csv` giữ dự đoán đã hiệu chỉnh của model 90/10;
`submission_selection_raw.csv` giữ dự đoán raw. CSV được lưu trước khi train full-data.
`submission.csv` sẽ được cập nhật bằng model refit khi bước tiếp theo hoàn tất.


In [ ]:
def validate_submission(frame, expected_test, template):
    assert list(frame.columns) == ["id", "label"], "Sai cot submission."
    assert len(frame) == len(expected_test) == 6000, "Sai so dong."
    assert frame["id"].is_unique, "ID trung."
    assert frame.notna().all().all(), "Submission co gia tri thieu."
    assert set(frame["id"]) == set(expected_test["id"]), "Thieu/thua ID."
    assert frame["id"].tolist() == template["id"].tolist(), "Sai thu tu sample."
    assert frame["label"].isin(LABELS).all(), "Nhan khong hop le."

def make_submission(predicted, destination):
    assert len(predicted) == len(test_df)
    predictions = pd.DataFrame({"id": test_df["id"].to_numpy(),
                                "label": [ID2LABEL[int(i)] for i in predicted]})
    result = sample_df[["id"]].merge(predictions, on="id", how="left", sort=False, validate="one_to_one")
    validate_submission(result, test_df, sample_df)
    result.to_csv(destination, index=False, encoding="utf-8")
    validate_submission(pd.read_csv(destination, dtype=str, keep_default_na=False), test_df, sample_df)
    return result

# Luu selection CSV truoc full refit de co ket qua neu runtime bi ngat.
make_submission(test_logits.argmax(axis=1), ARTIFACT_DIR / "submission_selection_raw.csv")
SUBMISSION_PATH = Path("/content/submission.csv")
submission = make_submission(final_test_pred, SUBMISSION_PATH)
shutil.copy2(SUBMISSION_PATH, ARTIFACT_DIR / "submission_selection.csv")
shutil.copy2(SUBMISSION_PATH, "/content/submission_selection.csv")

final_report = pd.DataFrame(classification_report(
    valid_true, final_valid_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    output_dict=True, zero_division=0,
)).T
print("Selection fitted validation report (NOT an independent score):")
display(final_report)
final_report.to_csv(ARTIFACT_DIR / "validation_final_fitted_report.csv")
final_language_rows = []
for language in LANGUAGES:
    mask = valid_df["language"].eq(language).to_numpy()
    scores = f1_score(valid_true[mask], final_valid_pred[mask], labels=np.arange(len(LABELS)),
                      average=None, zero_division=0)
    row = {"language": language, "rows": int(mask.sum()), "macro_f1": float(scores.mean())}
    row.update({f"f1_{label}": float(s) for label, s in zip(LABELS, scores)})
    final_language_rows.append(row)
pd.DataFrame(final_language_rows).to_csv(ARTIFACT_DIR / "validation_final_by_language.csv", index=False)
final_matrix = confusion_matrix(valid_true, final_valid_pred, labels=np.arange(len(LABELS)))
pd.DataFrame(final_matrix, index=LABELS, columns=LABELS).to_csv(ARTIFACT_DIR / "validation_final_confusion.csv")
final_errors = raw_errors.copy()
final_errors["predicted_label"] = [ID2LABEL[int(i)] for i in final_valid_pred]
final_errors.loc[valid_true != final_valid_pred].to_csv(ARTIFACT_DIR / "validation_errors_final.csv", index=False)
final_valid_rows = valid_df[["id", "language", "label"]].copy()
final_valid_rows["predicted_label"] = [ID2LABEL[int(i)] for i in final_valid_pred]
final_valid_rows.to_csv(ARTIFACT_DIR / "validation_final_predictions.csv", index=False)
np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
(ARTIFACT_DIR / "postprocess.json").write_text(
    json.dumps(POSTPROCESS_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
(ARTIFACT_DIR / "postprocess_selection.json").write_text(
    json.dumps(POSTPROCESS_SELECTION_REPORT, indent=2, ensure_ascii=False), encoding="utf-8")
oof_report = valid_df[["id", "language", "label"]].copy()
oof_report["predicted_label"] = [ID2LABEL[int(i)] for i in postprocess_oof_pred]
oof_report.to_csv(ARTIFACT_DIR / "validation_postprocess_crossfit.csv", index=False)
display(submission.head())
display(submission["label"].value_counts().reindex(LABELS, fill_value=0))
print("Submission:", SUBMISSION_PATH)
print("SHA256:", hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest())


## 11. Refit cuối trên toàn bộ dữ liệu có nhãn

Giữ nguyên checkpoint selection và `submission_selection.csv`. Khởi tạo lại từ pretrained,
train đủ 14.000 mẫu trong số epoch tốt nhất đã chọn. Không early-stop/evaluate trên dữ liệu
đã học, không tune lại bias. `USE_FULL_TRAIN=False` bỏ qua bước này.
Các temperature/bias đã chọn có thể chuyển sang model mới không hoàn hảo; giữ cả hai CSV để đối chiếu.


In [ ]:
"""Notebook cell source: refit a fresh model on all labelled training rows.

Paste after the selection submission/artifacts cell. This source intentionally
depends on the notebook namespace and must not be run as a standalone script.
Validation predictions below always belong to the held-out selection stage.
"""

# Capture the selection state before replacing trainer/model. These Python and
# NumPy objects contain no live GPU model or optimizer references.
SELECTED_EPOCH = int(selected_checkpoint_epoch(trainer.state))
SELECTION_TRAINING_LOG = json.loads(json.dumps(trainer.state.log_history))
SELECTION_TRAINING_ARGUMENTS = json.loads(training_args.to_json_string())
SELECTION_TRAIN_METRICS = dict(train_result.metrics)
FROZEN_POSTPROCESS_CONFIG = json.loads(json.dumps(POSTPROCESS_CONFIG))
SELECTION_REPORT = {
    "training_rows": int(len(fit_df)),
    "validation_rows": int(len(valid_df)),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_macro_f1": float(trainer.state.best_metric),
    "selected_epoch": SELECTED_EPOCH,
    "adapter": str(ADAPTER_DIR.relative_to(ARTIFACT_DIR)),
    "lexical": "selection_lexical.joblib" if ENABLE_LEXICAL_BRANCH else None,
    "postprocess_selection": json.loads(json.dumps(POSTPROCESS_SELECTION_REPORT)),
}
(ARTIFACT_DIR / "selection_report.json").write_text(
    json.dumps(SELECTION_REPORT, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_training_log.json").write_text(
    json.dumps(SELECTION_TRAINING_LOG, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_training_arguments.json").write_text(
    json.dumps(SELECTION_TRAINING_ARGUMENTS, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_train_metrics.json").write_text(
    json.dumps(SELECTION_TRAIN_METRICS, indent=2), encoding="utf-8",
)

# Preserve the 90/10 model's predictions even when full-train predictions later
# replace the notebook's active test variables and submission.csv.
selection_test_logits = np.array(test_logits, dtype=np.float32, copy=True)
selection_test_probabilities = np.array(final_test_proba, copy=True)
np.save(ARTIFACT_DIR / "selection_test_logits.npy", selection_test_logits)
np.save(ARTIFACT_DIR / "selection_test_probabilities.npy", selection_test_probabilities)
if test_lexical_scores is not None:
    np.save(ARTIFACT_DIR / "selection_test_lexical_scores.npy", test_lexical_scores)

FINAL_FIT_REPORT = {
    "enabled": bool(USE_FULL_TRAIN),
    "completed": not USE_FULL_TRAIN,
    "training_rows": int(len(fit_df)),
    "epochs": SELECTED_EPOCH,
    "epoch_source": "holdout_best_checkpoint",
    "active_adapter": SELECTION_REPORT["adapter"],
    "active_lexical": SELECTION_REPORT["lexical"],
    "postprocess_frozen": True,
    "validation_evaluated_after_refit": False,
    "full_train_validation_macro_f1": None,
    "note": (
        "Validation and cross-fit scores refer only to the selection model. "
        "A refit on all labelled rows has no untouched validation score; "
        "its frozen temperatures/biases may transfer imperfectly."
    ),
}

if USE_FULL_TRAIN:
    refit_epochs = float(SELECTED_EPOCH if FULL_TRAIN_EPOCHS is None else FULL_TRAIN_EPOCHS)
    if not np.isfinite(refit_epochs) or refit_epochs <= 0:
        raise ValueError("FULL_TRAIN_EPOCHS phai la so duong hoac None.")
    assert len(train_df) == 14000
    assert set(train_df["id"]).isdisjoint(set(test_df["id"]))
    full_train_df = train_df.reset_index(drop=True).copy()
    full_train_dataset = PromptDataset(full_train_df, labelled=True)
    full_weight_values = compute_class_weights(
        full_train_df["label_id"].to_numpy(), len(LABELS),
        power=CLASS_WEIGHT_POWER, enabled=USE_CLASS_WEIGHTS,
    )
    full_class_weights = torch.tensor(full_weight_values, dtype=torch.float32)
    full_weighted_focal_loss = make_weighted_focal_loss(full_class_weights, FOCAL_GAMMA)
    full_train_df[["id", "language", "label"]].to_csv(
        ARTIFACT_DIR / "full_train_rows.csv", index=False,
    )
    FINAL_FIT_REPORT.update({
        "training_rows": int(len(full_train_df)),
        "epochs": refit_epochs,
        "epoch_source": "holdout_best_checkpoint" if FULL_TRAIN_EPOCHS is None else "fixed_config_override",
        "class_weights": np.asarray(full_weight_values).tolist(),
    })
    print(
        "Full refit:", len(full_train_df), "rows | epochs =", refit_epochs,
        "| fresh backbone/head (adapter neu bat) | khong evaluate/tune tren validation.",
    )

    # Release Accelerator's references as well as Trainer's model, optimizer and
    # callbacks before allocating another backbone on the same GPU.
    trainer.accelerator.free_memory()
    del trainer, model, train_result
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(SEED)
    model = build_model()
    training_args = TrainingArguments(
        output_dir=str(RUN_DIR / "full_train"),
        run_name=RUN_NAME + "_full_train",
        num_train_epochs=refit_epochs,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        max_grad_norm=1.0,
        fp16=not USE_BF16,
        bf16=USE_BF16,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="no",
        save_strategy="no",
        load_best_model_at_end=False,
        logging_strategy="steps",
        logging_steps=25,
        label_names=["labels"],
        remove_unused_columns=True,
        eval_accumulation_steps=16,
        dataloader_num_workers=0,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        push_to_hub=False,
        torch_compile=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_train_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_loss_func=full_weighted_focal_loss,
    )
    trainer.model_accepts_loss_kwargs = False
    # Explicitly start from the factory's fresh model, never resume the 90/10 fit.
    train_result = trainer.train(resume_from_checkpoint=False)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()
    ADAPTER_DIR = ARTIFACT_DIR / ("final_model" if MODEL_PRESET == "wolf" else "final_adapter")
    trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    trainer.model.config.to_json_file(str(ARTIFACT_DIR / "base_config.json"))

    test_result = trainer.predict(test_dataset, metric_key_prefix="test")
    test_logits = test_result.predictions
    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]
    test_logits = np.asarray(test_logits, dtype=np.float32)
    if test_logits.shape != (len(test_df), len(LABELS)) or not np.isfinite(test_logits).all():
        raise ValueError("Full-train test logits sai shape hoac co NaN/Inf.")

    # Refit the vocabulary, IDF and classifiers on labelled train.csv only.
    # No validation or test scores/labels enter model or postprocessor selection.
    test_lexical_scores = None
    if ENABLE_LEXICAL_BRANCH:
        lexical_bundle = fit_lexical_branch(full_train_df)
        test_lexical_scores = lexical_predict(lexical_bundle, test_df)
        joblib.dump(lexical_bundle, ARTIFACT_DIR / "final_lexical.joblib", compress=3)
        np.save(ARTIFACT_DIR / "test_lexical_scores.npy", test_lexical_scores)
        FINAL_FIT_REPORT["active_lexical"] = "final_lexical.joblib"

    # Apply exactly the holdout-selected configuration; no postprocess fit call.
    if FROZEN_POSTPROCESS_CONFIG["enabled"]:
        final_test_scores = postprocess_log_scores(
            test_logits, test_lexical_scores, FROZEN_POSTPROCESS_CONFIG["parameters"],
        )
    else:
        final_test_scores = log_softmax_np(test_logits, 1.0)
    final_test_proba = softmax_from_log_scores(final_test_scores)
    final_test_pred = final_test_proba.argmax(axis=1)
    assert POSTPROCESS_CONFIG == FROZEN_POSTPROCESS_CONFIG
    np.testing.assert_allclose(final_test_proba.sum(axis=1), 1.0, atol=1e-6)
    assert np.isfinite(final_test_proba).all()
    np.save(ARTIFACT_DIR / "test_logits.npy", test_logits)
    np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
    FINAL_FIT_REPORT.update({
        "completed": True,
        "active_adapter": str(ADAPTER_DIR.relative_to(ARTIFACT_DIR)),
        "train_metrics": dict(train_result.metrics),
        "optimizer_steps": int(trainer.state.global_step),
    })
else:
    print("USE_FULL_TRAIN=False: giu model va submission tu holdout selection.")

(ARTIFACT_DIR / "final_fit_report.json").write_text(
    json.dumps(FINAL_FIT_REPORT, indent=2, ensure_ascii=False), encoding="utf-8",
)
print("Active model/adapter:", FINAL_FIT_REPORT["active_adapter"])
print("Active lexical model:", FINAL_FIT_REPORT["active_lexical"])
print("Validation reports van thuoc selection stage; chay cell xuat CSV tiep theo.")


In [ ]:
# Xuat du doan cua active model; final_valid_* van la selection predictions.
submission = make_submission(final_test_pred, SUBMISSION_PATH)
shutil.copy2(SUBMISSION_PATH, ARTIFACT_DIR / "submission.csv")
make_submission(test_logits.argmax(axis=1), ARTIFACT_DIR / "submission_raw.csv")
np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
print("Submission:", SUBMISSION_PATH, "| rows:", len(submission))
print("Weights:", FINAL_FIT_REPORT["active_adapter"])
print("SHA256:", hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest())
display(submission["label"].value_counts().reindex(LABELS, fill_value=0))


## 12. Lưu cấu hình và đóng gói

ZIP chứa selection và final weights: full model cho Wolf, adapter + heads cho DistilBERT.
Vì Wolf lưu toàn bộ weights, ZIP lớn hơn bản QLoRA. Bao gồm tokenizer, lexical joblib của từng
stage, postprocess.json, logits/ID, split, cấu hình và CSV. Đọc active_weights/active_lexical
trong run_config.json để tái lập submission; không ghép nhầm các stage.
DistilBERT cần tải lại đúng pretrained revision khi nạp adapter. Checkpoint chứa optimizer
ở RUN_DIR không nằm trong ZIP. Validation chỉ thuộc selection; refit dùng đủ 14.000 mẫu.


In [ ]:
run_config = {
    "team": "RMIT_SGS_Sigma69_C1",
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_source": MODEL_SOURCE,
    "requested_revision": MODEL_REVISION,
    "resolved_model_revision": RESOLVED_MODEL_REVISION,
    "labels": LABELS,
    "label2id": LABEL2ID,
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "notebook_version": "v3",
    "model_preset": MODEL_PRESET,
    "model_provenance": MODEL_PROVENANCE,
    "selection": SELECTION_REPORT,
    "full_refit": FINAL_FIT_REPORT,
    "active_weights": FINAL_FIT_REPORT["active_adapter"],
    "active_lexical": FINAL_FIT_REPORT["active_lexical"],
    "weights_format": "full_model" if MODEL_PRESET == "wolf" else "peft_adapter",
    "modernbert_reference_compile": False if MODEL_PRESET == "wolf" else None,
    "validation_size": VAL_SIZE,
    "training_rows": FINAL_FIT_REPORT["training_rows"],
    "validation_rows": len(valid_df),
    "test_rows": len(test_df),
    "max_length": MAX_LENGTH,
    "long_text_mode": LONG_TEXT_MODE,
    "epochs_requested": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "class_weights": FINAL_FIT_REPORT.get("class_weights", weight_values.tolist()),
    "class_weight_power": power,
    "postprocess": POSTPROCESS_CONFIG,
    "lexical": {"enabled": ENABLE_LEXICAL_BRANCH, "analyzer": "char_wb",
                "ngram_range": [2, 6], "C": LEXICAL_C, "class_weight": "balanced",
                "min_df": 2, "sublinear_tf": True, "max_features": LEXICAL_MAX_FEATURES,
                "global_weight": LEXICAL_GLOBAL_WEIGHT},
    "loss": "weighted_focal_mean_over_samples" if FOCAL_GAMMA else "weighted_cross_entropy_mean_over_samples",
    "focal_gamma": FOCAL_GAMMA,
    "quantization": {
        "load_in_4bit": True, "quant_type": "nf4", "double_quant": True,
        "compute_dtype": str(COMPUTE_DTYPE), "skip_modules": HEAD_MODULES,
    } if MODEL_PRESET == "distilbert" else None,
    "lora": {
        "rank": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGETS, "modules_to_save": HEAD_MODULES,
    } if MODEL_PRESET == "distilbert" else None,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capability": list(GPU_CAPABILITY),
    "package_versions": PACKAGE_VERSIONS,
    "data_sha256": DATA_HASHES,
    "best_checkpoint": SELECTION_REPORT["best_checkpoint"],
    "best_validation_macro_f1": SELECTION_REPORT["best_validation_macro_f1"],
    "baseline_validation_macro_f1": 0.8863657407407408,
    "baseline_public_score_user_reported": 0.87,
    "submission_sha256": hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest(),
}
(ARTIFACT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "label_mapping.json").write_text(
    json.dumps({"label2id": LABEL2ID, "id2label": ID2LABEL}, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "validation_metrics.json").write_text(
    json.dumps({"stage": "selection_only", "metrics": valid_result.metrics}, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "training_arguments.json").write_text(
    training_args.to_json_string(), encoding="utf-8",
)
(ARTIFACT_DIR / "training_log.json").write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "requirements-used.txt").write_text(
    "\n".join(f"{name}=={release}" for name, release in PACKAGE_VERSIONS.items()) + "\n",
    encoding="utf-8",
)
ARCHIVE_PATH = Path(shutil.make_archive(
    str(RUN_DIR / "rmit_c1_artifacts"), "zip", root_dir=str(ARTIFACT_DIR),
))
print("Artifacts:", ARTIFACT_DIR)
print("ZIP:", ARCHIVE_PATH, f"({ARCHIVE_PATH.stat().st_size / 2**20:.1f} MiB)")
PUBLIC_ARCHIVE_PATH = Path("/content/rmit_c1_artifacts.zip")
if ARCHIVE_PATH.resolve() != PUBLIC_ARCHIVE_PATH.resolve():
    shutil.copy2(ARCHIVE_PATH, PUBLIC_ARCHIVE_PATH)
print("Files panel download:", PUBLIC_ARCHIVE_PATH)


## 13. Tải kết quả

Mở panel **Files** bên trái Colab → Refresh → `submission.csv` → dấu ba chấm → **Download**.
ZIP `rmit_c1_artifacts.zip` cùng thư mục `/content` chứa model và báo cáo; chỉ CSV dùng để nộp dự đoán.
Không cần chạy lại train nếu trình duyệt chặn download. Hai cell dưới chỉ in đường dẫn;
đặt cờ `True` và chạy riêng một cell nếu muốn thử tải tự động.


In [ ]:
DOWNLOAD_CSV_NOW = False
print("Files panel: /content/submission.csv")
print("Size:", SUBMISSION_PATH.stat().st_size, "bytes")
if DOWNLOAD_CSV_NOW:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))


In [ ]:
DOWNLOAD_ZIP_NOW = False
print("Files panel:", PUBLIC_ARCHIVE_PATH)
if DOWNLOAD_ZIP_NOW:
    from google.colab import files
    files.download(str(PUBLIC_ARCHIVE_PATH))


## Nguồn và cách đọc kết quả

- File tham khảo trong repo: `rmit-mel-segfault-c1.ipynb`; xem `docs/melbourne-reference-review.md`.
- [Cuộc thi và quy định](https://www.kaggle.com/competitions/rmit-hackathon-2026/overview).
- [Wolf Defender trên Kaggle](https://www.kaggle.com/models/aisuko/wolf-defender-prompt-injection/transformers/default/1).
- [Trainer API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).
- [Focal loss](https://arxiv.org/abs/1708.02002).
- [KaggleHub model download](https://github.com/Kaggle/kagglehub#download-models).

Trong ZIP, `selection_report.json` và `validation_*` thuộc bước chọn mô hình;
`final_fit_report.json` ghi model nào tạo submission, số mẫu/epoch và đường dẫn weights.
Wolf lưu full model, DistilBERT lưu PEFT adapter: đọc `weights_format` trong `run_config.json`.
Khi load lại Wolf, đặt `reference_compile=False`, SDPA, đúng tokenizer và mapping nhãn.
Ensemble inference cần cả model/adapter, lexical joblib tương ứng và `postprocess.json`.
Không cộng các mức gain dự kiến để suy ra điểm 0.99; phải kiểm chứng từng lần chạy.
